# SMA20 > SMA50 Backtest Example

This notebook shows a simple long-only backtest:

- enter when `SMA20` crosses above `SMA50`
- exit when `SMA20` crosses below `SMA50`
- track `equity_list`
- plot price, moving averages, and equity with Plotly



## Imports

We import:

- `numpy` and `pandas` for data work
- `plotly.graph_objects` for interactive charts
- `polars` because the trade log is convenient as a DataFrame
- `backtester` the file for backtest and `SimpleBacktester` is the engine
- `yfinance` for stock data


In [1]:
import numpy as np
import pandas as pd
import polars as pl
import plotly.graph_objects as go
from backtester import SimpleBacktester
import yfinance as yf

## Get example price data

In [2]:

aapl = yf.Ticker("AAPL")
df = aapl.history(start='2024-03-07', end='2026-03-06')
print(df.head(5))

                                 Open        High         Low       Close  \
Date                                                                        
2024-03-07 00:00:00-05:00  167.624471  169.190223  166.970435  167.475830   
2024-03-08 00:00:00-05:00  167.475853  172.133462  167.416397  169.190247   
2024-03-11 00:00:00-04:00  171.380290  172.807306  170.498318  171.192001   
2024-03-12 00:00:00-04:00  171.588414  172.460482  169.467714  171.667694   
2024-03-13 00:00:00-04:00  171.211833  171.628043  169.219951  169.586624   

                             Volume  Dividends  Stock Splits  
Date                                                          
2024-03-07 00:00:00-05:00  71765100        0.0           0.0  
2024-03-08 00:00:00-05:00  76267000        0.0           0.0  
2024-03-11 00:00:00-04:00  60139500        0.0           0.0  
2024-03-12 00:00:00-04:00  59825400        0.0           0.0  
2024-03-13 00:00:00-04:00  52488700        0.0           0.0  


## Compute SMA20 and SMA50

This strategy uses two moving averages:

- `sma20`: faster trend
- `sma50`: slower trend

The rule is not simply “buy every bar where `sma20 > sma50`”.
Because your engine expects entry and exit events, we convert the condition into **cross signals**:

- entry when the condition changes from False to True
- exit when it changes from True to False


In [3]:
df['sma20'] = df['Close'].rolling(20).mean()
df['sma50'] = df['Close'].rolling(50).mean()

condition = df['sma20'] > df['sma50']
df['entries'] = condition & (~condition.shift(1, fill_value=False))
df['exits'] = (~condition) & (condition.shift(1, fill_value=False))

df.tail(10)


,Open,High,Low,Close,Volume,Dividends,Stock Splits,sma20,sma50,entries,exits
Date,,,,,,,,,,,
2026-02-20 00:00:00-05:00,258.970001,264.750000,258.160004,264.579987,42070500,0.0,0.0,264.895669,265.908597,False,False
2026-02-23 00:00:00-05:00,263.489990,269.429993,263.380005,266.179993,37308200,0.0,0.0,265.814262,265.693780,True,False
2026-02-24 00:00:00-05:00,267.859985,274.890015,267.709991,272.140015,47014600,0.0,0.0,266.662701,265.566193,False,False
2026-02-25 00:00:00-05:00,271.779999,274.940002,271.049988,274.230011,33714300,0.0,0.0,267.472775,265.495391,False,False
2026-02-26 00:00:00-05:00,274.950012,276.109985,270.799988,272.950012,32345100,0.0,0.0,268.310262,265.393994,False,False
2026-02-27 00:00:00-05:00,272.809998,272.809998,262.890015,264.179993,72366500,0.0,0.0,268.617334,265.200519,False,False
2026-03-02 00:00:00-05:00,262.410004,266.529999,260.200012,264.720001,41827900,0.0,0.0,268.891463,265.007854,False,False
2026-03-03 00:00:00-05:00,263.480011,265.559998,260.130005,263.750000,38568900,0.0,0.0,268.591083,264.851136,False,False
2026-03-04 00:00:00-05:00,264.649994,266.149994,261.420013,262.519989,39803100,0.0,0.0,268.255678,264.662825,False,False


## Run the backtest

We now pass close prices plus entry and exit signals into the backtester.

Settings used here:

- initial cash = 10,000
- position size = 2,000 USD per trade
- fee = 0.1%
- slippage = 0.05%


In [4]:
bt = SimpleBacktester(
    close=df['Close'].to_numpy(),
    entries=df['entries'].to_numpy(),
    exits=df['exits'].to_numpy(),
    fees=0.001,
    slippage=0.0005,
    size_usd=2_000,
    init_cash=10_000,
)

bt.summary()


col_names,column_0
str,f64
"""Start""",10000.0
"""End""",10443.68
"""win_rate[%]""",33.33
"""total_return[%]""",4.44
"""max_drawdown[%]""",-5.44
"""sharpe_per_period""",0.037
"""profit_factor""",2.3814
"""closed_trades""",6.0


## Inspect trade log and equity list

This lets you verify that the engine is doing what you expect.

- `trades` shows entries and exits
- `equity_list` is the mark-to-market account value at every bar


In [5]:
trades_df = bt.trades.to_pandas()
trades_df.head(10), bt.equity_list[:10]


(   index   type       price   pos_size  size_usd          cash        equity  \
 0     49  entry  188.383041  10.611361    2000.0   7998.000000   9997.000500   
 1    114   exit  225.031799  10.611361    2000.0  10382.313062  10385.893708   
 2    119  entry  226.541885   8.823978    2000.0   8380.313062  10379.313562   
 3    178   exit  227.039749   8.823978    2000.0  10380.702612  10383.706700   
 4    187  entry  241.331680   8.283208    2000.0   8378.702612  10377.703111   
 5    222   exit  228.611160   8.283208    2000.0  10269.496894  10272.336398   
 6    249  entry  234.308777   8.531479    2000.0   8267.496894  10266.497394   
 7    256   exit  213.071350   8.531479    2000.0  10082.584914  10085.310726   
 8    312  entry  200.021317   9.993937    2000.0   8080.584914  10079.585414   
 9    320   exit  195.046463   9.993937    2000.0  10026.944091  10029.867039   
 
    realized_pnl  
 0      0.000000  
 1    382.313062  
 2      0.000000  
 3     -1.610450  
 4      0.00

## Plot price, SMA20, SMA50, and trade markers

This chart helps you visually confirm whether the entries and exits line up with the moving-average crossover logic.


In [6]:
entry_points = df[df['entries']]
exit_points = df[df['exits']]

fig = go.Figure()

fig.add_trace(go.Scatter(x=df.index, y=df['Close'], mode='lines', name='Close'))
fig.add_trace(go.Scatter(x=df.index, y=df['sma20'], mode='lines', name='SMA20'))
fig.add_trace(go.Scatter(x=df.index, y=df['sma50'], mode='lines', name='SMA50'))

fig.add_trace(go.Scatter(
    x=entry_points.index, y=entry_points['Close'], mode='markers', name='Entries',
    marker=dict(symbol='triangle-up', size=10, color='limegreen')
))

fig.add_trace(go.Scatter(
    x=exit_points.index, y=exit_points['Close'], mode='markers', name='Exits',
    marker=dict(symbol='triangle-down', size=10, color='red')
))

fig.update_layout(
    title='SMA20 / SMA50 Crossover',
    xaxis_title='Date',
    yaxis_title='Price',
    template='plotly_white',
    height=550,
)

fig.show()


## Plot equity curve with Plotly

In [7]:
equity_df = pd.DataFrame({
    'date': df.index,
    'equity': bt.equity_list,
})

fig_eq = go.Figure()
fig_eq.add_trace(go.Scatter(x=df.index, y=equity_df['equity'], mode='lines', name='Equity'))

fig_eq.update_layout(
    title='Equity Curve',
    xaxis_title='Date',
    yaxis_title='Account Equity',
    template='plotly_white',
    height=500,
)

fig_eq.show()
